# Building a movie recommender with Naïve Bayes

## 1. Preparing the data

In [1]:
import numpy as np
import pandas as pd

In [4]:
data_path = "../../data/ml-1m/ratings.dat"
df = pd.read_csv(data_path, header=None, sep='::', engine='python')
df.columns = ['user_id', 'movie_id', 'rating', 'timestamp']

print(df)

         user_id  movie_id  rating  timestamp
0              1      1193       5  978300760
1              1       661       3  978302109
2              1       914       3  978301968
3              1      3408       4  978300275
4              1      2355       5  978824291
...          ...       ...     ...        ...
1000204     6040      1091       1  956716541
1000205     6040      1094       5  956704887
1000206     6040       562       5  956704746
1000207     6040      1096       4  956715648
1000208     6040      1097       4  956715569

[1000209 rows x 4 columns]


In [8]:
# How many unique users and movies are in this million-row dataset:
n_users = df['user_id'].nunique()
n_movies = df['movie_id'].nunique()

print(
  f"Number of users: {n_users}",
  f"\nNumber of movies: {n_movies}"
)

Number of users: 6040 
Number of movies: 3706


In [9]:
# Construct a 6,040 (no. of users) by 3,706 (no. of movies) matrix where each row contains movie ratings from a user, and each column represents a movie:
def load_user_rating_data(df, n_users, n_movies):
  data = np.zeros([n_users, n_movies], dtype=np.intc)
  movie_id_mapping = {}

  for user_id, movie_id, rating in zip(df['user_id'], df['movie_id'], df['rating']):
    user_id = int(user_id) - 1
    
    if movie_id not in movie_id_mapping:
      movie_id_mapping[movie_id] = len(movie_id_mapping)
    
    data[user_id, movie_id_mapping[movie_id]] = rating

  return data, movie_id_mapping

data, movie_id_mapping = load_user_rating_data(df, n_users, n_movies)

In [10]:
# Always recommended to analyze the data distribution in order to identify if there is a class imbalance issue in the dataset:
values, counts = np.unique(data, return_counts=True)

for value, count in zip(values, counts):
  print(f"Number of rating {value}: {count}")

Number of rating 0: 21384031
Number of rating 1: 56174
Number of rating 2: 107557
Number of rating 3: 261197
Number of rating 4: 348971
Number of rating 5: 226310


In [11]:
# Since most ratings are unknown, we take the movie with the most known ratings as our target movie for easier prediction validation:
print(df['movie_id'].value_counts())

movie_id
2858    3428
260     2991
1196    2990
1210    2883
480     2672
        ... 
3458       1
2226       1
1815       1
398        1
2909       1
Name: count, Length: 3706, dtype: int64


In [13]:
# The target movie is ID, and we will treat ratings of other movies as features. We only use rows with ratings available for the target movie so we can validate how good the prediction is:
target_movie_id = 2858
X_raw = np.delete(data, movie_id_mapping[target_movie_id], axis=1)
Y_raw = data[:, movie_id_mapping[target_movie_id]]
X = X_raw[Y_raw > 0]
Y = Y_raw[Y_raw > 0]

print(
  f"Shape of X: {X.shape}",
  f"\nShape of Y: {Y.shape}"
)

Shape of X: (3428, 3705) 
Shape of Y: (3428,)
